In [12]:
from itertools import chain

import networkx as nx
import numpy as np
import pandas as pd
from networkx.drawing.nx_agraph import graphviz_layout

In [2]:
A =[[  0, 1, 0, 0, 0],
    [  0, 0, 1, 0, 0],
    [  0, 0, 0, 1, 0],
    [  0, 0, 0, 0, 1],
    [0.1, 0, 0, 0, 0]]

In [3]:
G = nx.DiGraph()
edges = chain.from_iterable(
    [(i, j) for j, col in enumerate(row) if A[i][j]] 
    for i, row in enumerate(A))

In [4]:
G.add_edges_from(edges)
print(G.edges(data=True))

[(0, 1, {}), (1, 2, {}), (2, 3, {}), (3, 4, {}), (4, 0, {})]


In [5]:
G = nx.DiGraph()
edges = chain.from_iterable(
    [(i, j, {'weight': A[i][j]}) for j, col in enumerate(row) if A[i][j]]
    for i, row in enumerate(A))
G.add_edges_from(edges)
print(G.edges(data=True))

[(0, 1, {'weight': 1}), (1, 2, {'weight': 1}), (2, 3, {'weight': 1}), (3, 4, {'weight': 1}), (4, 0, {'weight': 0.1})]


In [6]:
A_mtx = np.matrix(A)
G = nx.from_numpy_matrix(A_mtx, create_using=nx.DiGraph())
print(G.edges(data=True))

[(0, 1, {'weight': 1.0}), (1, 2, {'weight': 1.0}), (2, 3, {'weight': 1.0}), (3, 4, {'weight': 1.0}), (4, 0, {'weight': 0.1})]


In [7]:
B_mtx = nx.to_numpy_matrix(G)
B_mtx

matrix([[0. , 1. , 0. , 0. , 0. ],
        [0. , 0. , 1. , 0. , 0. ],
        [0. , 0. , 0. , 1. , 0. ],
        [0. , 0. , 0. , 0. , 1. ],
        [0.1, 0. , 0. , 0. , 0. ]])

In [8]:
B_mtx.tolist()

[[0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 1.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 1.0],
 [0.1, 0.0, 0.0, 0.0, 0.0]]

In [9]:
labels = 'Born', 'Married', 'Elected Rep', 'Elected Pres', 'Died'
nx.relabel_nodes(G, dict(enumerate(labels)), copy=False)
adj = nx.convert_matrix.to_pandas_adjacency(G)
print(type(adj))
adj;

<class 'pandas.core.frame.DataFrame'>


,Born,Married,Elected Rep,Elected Pres,Died
Born,0.0,1.0,0.0,0.0,0.0
Married,0.0,0.0,1.0,0.0,0.0
Elected Rep,0.0,0.0,0.0,1.0,0.0
Elected Pres,0.0,0.0,0.0,0.0,1.0
Died,0.1,0.0,0.0,0.0,0.0


In [10]:
edges = nx.to_pandas_edgelist(G)
edges

,source,target,weight
0,Born,Married,1.0
1,Married,Elected Rep,1.0
2,Elected Rep,Elected Pres,1.0
3,Elected Pres,Died,1.0
4,Died,Born,0.1


In [15]:
edges = pd.DataFrame({'from': {0: 'Died', 
                               1: 'Elected Rep', 
                               2: 'Married', 
                               3: 'Born', 
                               4: 'Elected Pres'},
                      'to': {0: 'Born', 
                             1: 'Elected Pres',
                             2: 'Elected Rep',
                             3: 'Married',
                             4: 'Died'},
                      'weight': {0: 0.1, 1: 1, 2: 1, 3: 1, 4: 1}})
edges

,from,to,weight
0,Died,Born,0.1
1,Elected Rep,Elected Pres,1.0
2,Married,Elected Rep,1.0
3,Born,Married,1.0
4,Elected Pres,Died,1.0


In [16]:
G = nx.from_pandas_edgelist(edges, *edges.columns)
G.edges(data=True)

EdgeDataView([('Died', 'Born', {'weight': 0.1}), ('Died', 'Elected Pres', {'weight': 1.0}), ('Born', 'Married', {'weight': 1.0}), ('Elected Rep', 'Elected Pres', {'weight': 1.0}), ('Elected Rep', 'Married', {'weight': 1.0})])

In [18]:
events = {'Born': 1809, 
          'Elected Rep': 1847, 
          'Married': 1842,
          'Elected Pres': 1861,
          'Died': 1865}
nx.set_node_attributes(G, events, 'date')
node_data = G.nodes(data=True)
node_data

NodeDataView({'Died': {'date': 1865}, 'Born': {'date': 1809}, 'Elected Rep': {'date': 1847}, 'Elected Pres': {'date': 1861}, 'Married': {'date': 1842}})

In [20]:
lincoln_ser = pd.DataFrame(list(node_data)).set_index(0)[1]
lincoln_ser

0
Died            {'date': 1865}
Born            {'date': 1809}
Elected Rep     {'date': 1847}
Elected Pres    {'date': 1861}
Married         {'date': 1842}
Name: 1, dtype: object

In [21]:
df = lincoln_ser.apply(pd.Series)
df

,date
0,
Died,1865
Born,1809
Elected Rep,1847
Elected Pres,1861
Married,1842


In [22]:
spans = df.sort_values('date').diff()
spans

,date
0,
Born,NaN
Married,33.0
Elected Rep,5.0
Elected Pres,14.0
Died,4.0
